# LASSER on Kaggle

GEARS with an optional GSR-style learned co-expression graph. Needs **Internet on** (GEARS downloads Norman and the GO graph from Harvard Dataverse) and a **GPU (T4)**.

Cells: install, get the code, run the tests (before any real training), baseline run, graph-learning run, results.

In [ ]:
# 1. Dependencies. GEARS is vendored inside lasser/ (lasser/gears), so no cell-gears install.
#    Kaggle already has torch, numpy, pandas, scipy, scikit-learn, networkx, tqdm, requests.
!pip install -q torch_geometric scanpy pytest

In [ ]:
# 2. Get the code: clone, then move the self-contained package next to the notebook.
REPO_URL = "https://github.com/<me>/<lasser-repo>.git"   # <- set this
REPO_DIR = "/kaggle/working/lasser-repo"
!rm -rf {REPO_DIR} /kaggle/working/lasser
!git clone -q {REPO_URL} {REPO_DIR}
!mv {REPO_DIR}/lasser /kaggle/working/lasser
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
# 3. Tests 1-8 (tiny subsets; first run also downloads Norman, ~10-20 min in total).
%env LASSER_DATA_DIR=/kaggle/working/data
%env LASSER_TEST_DIR=/kaggle/working/lasser_tests
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests -m "not smoke" -p no:cacheprovider -rA -s --tb=short 2>&1 | tee /kaggle/working/test_output.txt | grep -vE "it/s\]|s/it\]"

In [ ]:
# 3b. Optional test 9: full Norman, 1 epoch, flag off and on, checks the T4 memory budget.
%env LASSER_SMOKE=1
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests/test_09_smoke.py -p no:cacheprovider -rA -s --tb=short 2>&1 | tee /kaggle/working/smoke_output.txt | grep -vE "it/s\]|s/it\]"
%env LASSER_SMOKE=0

In [ ]:
# 4. Load the data once and reuse it for both runs.
from lasser import GSRConfig, LasserConfig, load_pert_data, load_run, run_experiment

common = dict(dataset="norman", seed=1, epochs=20,
              data_dir="/kaggle/working/data", out_dir="/kaggle/working/runs",
              cache_dir="/kaggle/working/cache", repo_dir=REPO_DIR)
pert_data = load_pert_data(LasserConfig(**common))

In [ ]:
# 5. Baseline: graph_learning=False is plain GEARS (plus logging and saving).
base = run_experiment(LasserConfig(**common, graph_learning=False), pert_data)
base["run_dir"]

In [ ]:
# 6. Graph learning: GSR-refined co-expression graph (defaults from the plan, section 11).
gl = run_experiment(LasserConfig(**common, graph_learning=True, gsr=GSRConfig()), pert_data)
gl["run_dir"], gl["checks"]["h2"], gl["checks"]["refine"]

In [ ]:
# 7. Results: the global run table, and any past run loaded back.
import pandas as pd
pd.set_option("display.max_columns", 200)
runs = pd.read_csv("/kaggle/working/runs/runs.csv")
cols = ["run_id", "seed", "graph_learning", "gsr_init_gene_emb", "graph_num_edges", "graph_isolated_genes",
        "test_pearson_delta", "test_mse_top20_de", "test_nmse_top20_de",
        "test_unseen_single_pearson_delta", "test_combo_seen0_pearson_delta",
        "test_combo_seen1_pearson_delta", "test_combo_seen2_pearson_delta", "run_dir"]
display(runs[[c for c in cols if c in runs.columns]])

r = load_run(gl["run_dir"])
display(r["epoch_log"])
r["test_metrics"]["paper"]["subgroups"]